In [4]:
import pandas as pd
from pathlib import Path
import os
import seaborn as sns
from seaborn import heatmap

In [5]:
NOTEBOOK_DIR = Path(os.getcwd()).resolve()
PROJECT_DIR = NOTEBOOK_DIR.parent
RAW_DATA_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_DIR / "data" / "processed"

In [6]:
df_trends = pd.read_parquet(RAW_DATA_DIR / "google_trends.parquet")

In [7]:
df_trends.head()

,date,region,brand,trend_index
0,2026-01-01,AU,Adidas,75
1,2026-01-02,AU,Adidas,72
2,2026-01-03,AU,Adidas,68
3,2026-01-04,AU,Adidas,65
4,2026-01-05,AU,Adidas,64


In [8]:
df_trends.info()

<class 'pandas.DataFrame'>
RangeIndex: 15750 entries, 0 to 15749
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   date         15750 non-null  datetime64[ms]
 1   region       15750 non-null  str           
 2   brand        15750 non-null  str           
 3   trend_index  15750 non-null  int64         
dtypes: datetime64[ms](1), int64(1), str(2)
memory usage: 624.6 KB


In [9]:
pd.Series(df_trends[['region', 'brand']].value_counts().to_list()).value_counts()

90    175
Name: count, dtype: int64

In [10]:
df_retail = pd.read_csv(RAW_DATA_DIR / "retail_pricing_demand_100k.csv")
df_retail.head()

,date,product_id,category,brand,region,channel,season,base_price,current_price,price_change_pct,discount_pct,promotion_type,units_sold,revenue,inventory_level,stockout_flag,demand_index
0,2026-01-01,P1001,Shoes,Nike,AU,mobile,Winter,173.55,112.81,-35.0,35,Buy One Get One,15,1692.15,153,0,133.93
1,2026-01-02,P1001,Shoes,Nike,AU,web,Winter,173.55,173.55,0.0,0,NaN,10,1735.50,185,0,89.29
2,2026-01-03,P1001,Shoes,Nike,AU,app,Winter,173.55,173.55,0.0,0,NaN,13,2256.15,312,0,116.07
3,2026-01-04,P1001,Shoes,Nike,AU,web,Winter,173.55,164.87,-5.0,5,Member Offer,13,2143.31,227,0,116.07
4,2026-01-05,P1001,Shoes,Nike,AU,app,Winter,173.55,173.55,0.0,0,NaN,13,2256.15,247,0,116.07


In [11]:
df_grouped = df_trends.groupby(['region', 'brand'])['trend_index'].transform('mean')
df_grouped = df_trends.groupby(['region', 'brand'])['trend_index'].transform('std')
df_grouped

0         9.972897
1         9.972897
2         9.972897
3         9.972897
4         9.972897
           ...    
15745    13.809819
15746    13.809819
15747    13.809819
15748    13.809819
15749    13.809819
Name: trend_index, Length: 15750, dtype: float64

In [44]:
df_master = pd.read_parquet(PROCESSED_DATA_DIR / "master_dataset_simulated.parquet")
df_master.head()

,date,product_id,category,brand,region,channel,season,base_price,current_price,price_change_pct,discount_pct,promotion_type,revenue,inventory_level,stockout_flag,demand_index,trend_momentum,units_sold
0,2026-01-01,P1001,Shoes,Nike,AU,mobile,Winter,173.55,112.81,-35.0,35,Buy One Get One,1692.15,153,0,133.93,0.000000,17.0
1,2026-01-02,P1001,Shoes,Nike,AU,web,Winter,173.55,173.55,0.0,0,NaN,1735.50,185,0,89.29,0.000000,11.0
2,2026-01-03,P1001,Shoes,Nike,AU,app,Winter,173.55,173.55,0.0,0,NaN,2256.15,312,0,116.07,0.000000,14.0
3,2026-01-04,P1001,Shoes,Nike,AU,web,Winter,173.55,164.87,-5.0,5,Member Offer,2143.31,227,0,116.07,0.000000,12.0
4,2026-01-05,P1001,Shoes,Nike,AU,app,Winter,173.55,173.55,0.0,0,NaN,2256.15,247,0,116.07,0.113504,13.0


In [45]:
num_cols = df_master.select_dtypes('number')
num_cols

,base_price,current_price,price_change_pct,discount_pct,revenue,inventory_level,stockout_flag,demand_index,trend_momentum,units_sold
0,173.55,112.81,-35.0,35,1692.15,153,0,133.93,0.000000,17.0
1,173.55,173.55,0.0,0,1735.50,185,0,89.29,0.000000,11.0
2,173.55,173.55,0.0,0,2256.15,312,0,116.07,0.000000,14.0
3,173.55,164.87,-5.0,5,2143.31,227,0,116.07,0.000000,12.0
4,173.55,173.55,0.0,0,2256.15,247,0,116.07,0.113504,13.0
...,...,...,...,...,...,...,...,...,...,...
172795,276.02,276.02,0.0,0,1932.14,348,0,83.33,-0.385315,7.0
172796,276.02,234.62,-15.0,15,2346.20,301,0,119.05,-0.491784,10.0
172797,276.02,276.02,0.0,0,2484.18,289,0,107.14,0.060839,9.0
172798,276.02,276.02,0.0,0,2760.20,266,0,119.05,0.588113,11.0


In [46]:
num_cols.corr()

,base_price,current_price,price_change_pct,discount_pct,revenue,inventory_level,stockout_flag,demand_index,trend_momentum,units_sold
base_price,1.000000,0.949737,0.002804,-0.002805,0.661226,-0.010677,0.002887,0.003651,-0.001257,0.036174
current_price,0.949737,1.000000,0.274780,-0.274781,0.642556,-0.002322,0.002013,-0.142502,-0.000685,-0.036299
price_change_pct,0.002804,0.274780,1.000000,-1.000000,0.043455,0.026723,-0.002997,-0.532908,-0.000784,-0.252093
discount_pct,-0.002805,-0.274781,-1.000000,1.000000,-0.043455,-0.026723,0.002997,0.532910,0.000783,0.252093
revenue,0.661226,0.642556,0.043455,-0.043455,1.000000,-0.068256,0.010948,0.266217,-0.003439,0.631034
inventory_level,-0.010677,-0.002322,0.026723,-0.026723,-0.068256,1.000000,-0.112585,-0.057531,0.002032,-0.097886
stockout_flag,0.002887,0.002013,-0.002997,0.002997,0.010948,-0.112585,1.000000,0.009402,0.003849,0.016536
demand_index,0.003651,-0.142502,-0.532908,0.532910,0.266217,-0.057531,0.009402,1.000000,-0.012793,0.564084
trend_momentum,-0.001257,-0.000685,-0.000784,0.000783,-0.003439,0.002032,0.003849,-0.012793,1.000000,0.034153
units_sold,0.036174,-0.036299,-0.252093,0.252093,0.631034,-0.097886,0.016536,0.564084,0.034153,1.000000


In [47]:
y = df_master['units_sold']
X = num_cols

In [48]:
from sklearn.feature_selection import mutual_info_regression

In [49]:
mi = mutual_info_regression(X, y, random_state=42)

In [53]:
nums = num_cols.columns.to_list()

In [54]:
mi

array([7.85565764e-01, 6.51670554e-01, 4.63862194e-02, 4.30786177e-02,
       1.01102650e+00, 5.58286878e-03, 1.25123436e-03, 1.19112798e+00,
       1.00816059e-02, 3.47191499e+00])

In [55]:
dictionary = dict(zip(nums, mi))

In [56]:
dictionary

{'base_price': np.float64(0.7855657636063365),
 'current_price': np.float64(0.651670553845296),
 'price_change_pct': np.float64(0.046386219438442566),
 'discount_pct': np.float64(0.043078617675067044),
 'revenue': np.float64(1.0110264959780793),
 'inventory_level': np.float64(0.005582868782903816),
 'stockout_flag': np.float64(0.0012512343629591527),
 'demand_index': np.float64(1.1911279755453537),
 'trend_momentum': np.float64(0.010081605885983613),
 'units_sold': np.float64(3.4719149862203444)}